In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_info_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateInfoId String, SourceSystem String,"
        #     "IngestionTimestamp timestamp, BatchId String, RealEstateInfoKey string, RealEstateKey String, OwnerPartyKey String, OwnershipKey String, OwnershipReferenceKey String,"
        #     "OwnershipCode String, OwnershipTypeCode String, OwnershipType String, OwnershipPercentage Double,"
        #     "OwnershipStatusCode String, OwnershipStatus String, OwnershipStatusDate Date, NotaryDate Date, NotaryNumber String,"
        #     "RELienKey String, RELienReferenceKey String, RELienIsActiveFlag Boolean, RELienStatusCode String, RELienStatus String,"
        #     "RELienActive Boolean, RELienTypeCode String, RELienType String, RELienReasonCode String,"
        #     "RELienReason String, RELienInitialOrder Int, RELienCurrentOrder Int, RELienCreditorKey String, RELienCreditorName String,"
        #     "RELienCreditorType string, RELienMortgageTypeCode String, RELienMortgageType String, RELienMortgageAmount Decimal(19,4), "
        #     "RELienCurrency String, RELienComments String, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateInfo"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== fsad_real_estate_investigations_properties_liens ==================

        fsad_real_estate_investigations_properties_liens_extra_filters = None

        if fsad_real_estate_investigations_properties_liens_extra_filters is not None:
            fsad_real_estate_investigations_properties_liens_extra_filters.append(
                date_filter
            )
            fsad_real_estate_investigations_properties_liens_final_filters = (
                " AND ".join(
                    fsad_real_estate_investigations_properties_liens_extra_filters
                )
            )
        else:
            fsad_real_estate_investigations_properties_liens_final_filters = date_filter

        fsad_real_estate_investigations_properties_liens_selected_fields = [
            "REIP_CODE",
            "REPL_UNIQUE_ID",
            "REPL_DATE",
            "REPL_ACTIVE",
            "REPL_ORDER",
            "REPL_CREDITOR",
            "REPL_CREDITOR_TYPE",
        ]

        fsad_real_estate_investigations_properties_liens = (
            load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "qca_fsad_real_estate_investigations_properties_liens",
                fsad_real_estate_investigations_properties_liens_final_filters,
                fsad_real_estate_investigations_properties_liens_selected_fields,
            )
        )

        # ================== fs_real_estate_investigations ==================

        fs_real_estate_investigations_extra_filters = None

        if fs_real_estate_investigations_extra_filters is not None:
            fs_real_estate_investigations_extra_filters.append(date_filter)
            fs_real_estate_investigations_final_filters = " AND ".join(
                fs_real_estate_investigations_extra_filters
            )
        else:
            fs_real_estate_investigations_final_filters = date_filter

        fs_real_estate_investigations_selected_fields = [
            "CUST_CODE",
            "REIN_CODE",
            "businessdate",
        ]

        fs_real_estate_investigations = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_real_estate_investigations",
            fs_real_estate_investigations_final_filters,
            fs_real_estate_investigations_selected_fields,
        )

        fs_real_estate_investigations = (
            fs_real_estate_investigations.withColumnRenamed("REIP_CODE", "REIP_CODE_FS")
        ).withColumnRenamed("REIN_CODE", "REIN_CODE_FS")

        # ================== fsad_real_estate_investigations_properties ==================

        fsad_real_estate_investigations_properties_extra_filters = None

        if fsad_real_estate_investigations_properties_extra_filters is not None:
            fsad_real_estate_investigations_properties_extra_filters.append(date_filter)
            fsad_real_estate_investigations_properties_final_filters = " AND ".join(
                fsad_real_estate_investigations_properties_extra_filters
            )
        else:
            fsad_real_estate_investigations_properties_final_filters = date_filter

        fsad_real_estate_investigations_properties_selected_fields = [
            "REIP_CODE",
            "REIP_COOWNERSHIP_TYPE_INTCODE",
            "REIN_CODE",
        ]

        fsad_real_estate_investigations_properties = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_real_estate_investigations_properties",
            fsad_real_estate_investigations_properties_final_filters,
            fsad_real_estate_investigations_properties_selected_fields,
        )
        fsad_real_estate_investigations_properties = (
            fsad_real_estate_investigations_properties.withColumnRenamed(
                "REIP_CODE", "REIP_CODE_P"
            )
        )
        # ================== fsad_properties_real_estates ==================

        fsad_properties_real_estates_extra_filters = None

        if fsad_properties_real_estates_extra_filters is not None:
            fsad_properties_real_estates_extra_filters.append(date_filter)
            fsad_properties_real_estates_final_filters = " AND ".join(
                fsad_properties_real_estates_extra_filters
            )
        else:
            fsad_properties_real_estates_final_filters = date_filter

        fsad_properties_real_estates_selected_fields = [
            "pro_code",
            "cust_code",
            "PRO_ASSET_PERCENTAGE",
            "REST_CODE",
            "businessdate",
        ]

        fsad_properties_real_estates = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_properties_real_estates",
            fsad_properties_real_estates_final_filters,
            fsad_properties_real_estates_selected_fields,
        )

        # ================== at_property_liens ==================

        at_property_liens_extra_filters = None

        if at_property_liens_extra_filters is not None:
            at_property_liens_extra_filters.append(date_filter)
            at_property_liens_final_filters = " AND ".join(
                at_property_liens_extra_filters
            )
        else:
            at_property_liens_final_filters = date_filter

        at_property_liens_selected_fields = [
            "lie_code", 
            "PRO_CODE",
            ]

        at_property_liens = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_property_liens",
            at_property_liens_final_filters,
            at_property_liens_selected_fields,
        )

        at_property_liens = (
            at_property_liens.withColumnRenamed("lie_code", "lie_code_apl")
            .withColumnRenamed("PRO_CODE", "pro_code_apl")
                            )

        # ================== at_liens ==================

        at_liens_extra_filters = None

        if at_liens_extra_filters is not None:
            at_liens_extra_filters.append(date_filter)
            at_liens_final_filters = " AND ".join(at_liens_extra_filters)
        else:
            at_liens_final_filters = date_filter

        at_liens_selected_fields = [
            "LIE_DATE",
            "LIE_ACTIVE_FLAG",
            "LIE_ORDER",
            "LIE_LENDER",
            "LIE_LENDER_TYPE",
            "LIE_CURRENCY",
            "LIE_CODE",
            "LIE_TYPE",
            "LIE_VALUE",
        ]

        at_liens = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_liens",
            at_liens_final_filters,
            at_liens_selected_fields,
        )

        # ========================== at_properties ==========================

        at_properties_owner_extra_filters = None

        if at_properties_owner_extra_filters is not None:
            at_properties_owner_extra_filters.append(date_filter)
            at_properties_final_filters = " AND ".join(
                at_properties_owner_extra_filters
            )
        else:
            at_properties_final_filters = date_filter

        at_properties_selected_fields = [
            "PRO_CODE",
            "PRO_ASSET_TYPE_CODE",
            "PRO_ASSET_PERCENTAGE",
            "REST_CODE",
            "CUST_CODE",
        ]

        at_properties = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_properties",
            at_properties_final_filters,
            at_properties_selected_fields,
        )

        at_properties = (
                at_properties.withColumnRenamed("PRO_CODE", "PRO_CODE_atp")
                .withColumnRenamed("REST_CODE", "REST_CODE_atp")
                .withColumnRenamed("PRO_ASSET_PERCENTAGE", "PRO_ASSET_PERCENTAGE_atp")
                .withColumnRenamed("CUST_CODE", "CUST_CODE_atp")
                        )

        # ========================== at_real_estates ==========================

        at_real_estates_owner_extra_filters = None

        if at_real_estates_owner_extra_filters is not None:
            at_real_estates_owner_extra_filters.append(date_filter)
            at_real_estates_final_filters = " AND ".join(
                at_real_estates_owner_extra_filters
            )
        else:
            at_real_estates_final_filters = date_filter

        at_real_estates_selected_fields = [
            "REST_CODE",
            "REST_OWNERSHIP_TYPE_INTCODE",
        ]

        at_real_estates = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_real_estates",
            at_real_estates_final_filters,
            at_real_estates_selected_fields,
        )

        at_real_estates = (
                at_real_estates
                .withColumnRenamed("REST_CODE", "REST_CODE_atre")
                        )


        # ================== Joins ==================
        target_table = at_properties.join(
            at_property_liens,
            (at_properties["PRO_CODE_atp"]== at_property_liens["pro_code_apl"]),
            "left",
        ).join(
            at_real_estates,
            (at_properties["REST_CODE_atp"]== at_real_estates["REST_CODE_atre"]),
            "left",
        ).join(
            at_liens,
            (at_property_liens["lie_code_apl"] == at_liens["LIE_CODE"]),
            "left",
        ).join(
            fsad_properties_real_estates,
            (fsad_properties_real_estates["pro_code"]== at_property_liens["pro_code_apl"]),
            "left",
        )


        target_table_primary_key_fields = [
            target_table.pro_code_apl,
            target_table.CUST_CODE_atp,
            target_table.lie_code_apl,
            lit("qca"),
        ]
        RealEstateInfoKey_primary_key_fields = [
            target_table.pro_code_apl,
            target_table.CUST_CODE_atp,
            target_table.lie_code_apl,
        ]

        target_table = target_table.withColumn(
            "RealEstateInfoId",
            sha2(concat_ws(";", *target_table_primary_key_fields), 256),
        ).withColumn(
            "RealEstateInfoKey", concat_ws(";", *RealEstateInfoKey_primary_key_fields)
        )


        result_df = target_table.select(
            target_table.RealEstateInfoId,
            lit("qca").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            target_table.RealEstateInfoKey.cast("string"),
            target_table.REST_CODE_atp.cast("String").alias("RealEstateKey"),
            target_table.CUST_CODE_atp.cast("String").alias("OwnerPartyKey"),
            lit(None).cast("string").alias("OwnershipKey"),
            lit(None).cast("string").alias("OwnershipReferenceKey"),
            lit(None).alias("OwnershipCode"),
            target_table.REST_OWNERSHIP_TYPE_INTCODE.alias("OwnershipTypeCode"), #need descriptions to assign to codes
            target_table.REST_OWNERSHIP_TYPE_INTCODE.alias("OwnershipType"),     
            target_table.PRO_ASSET_PERCENTAGE_atp.cast("double").alias(
                "OwnershipPercentage"
            ), 
            lit(None).alias("OwnershipStatusCode"),
            lit(None).alias("OwnershipStatus"),
            lit(None).alias("OwnershipStatusDate"),
            target_table.LIE_DATE.cast("date").alias("NotaryDate"),
            lit(None).alias("NotaryNumber"),
            target_table.lie_code_apl.cast("string").alias("RELienKey"),
            lit(None).cast("string").alias("RELienReferenceKey"),
            target_table.LIE_ACTIVE_FLAG.alias("RELienIsActiveFlag"),
            lit(None).alias("RELienStatusCode"),
            lit(None).alias("RELienStatus"),
            lit(None).alias("RELienActive"),
            target_table.LIE_TYPE.alias("RELienTypeCode"),
            target_table.LIE_TYPE.alias("RELienType"),
            lit(None).alias("RELienReasonCode"),
            lit(None).alias("RELienReason"),
            target_table.LIE_ORDER.cast("int").alias("RELienInitialOrder"), 
            target_table.LIE_ORDER.cast("int").alias("RELienCurrentOrder"),
            target_table.LIE_LENDER.cast("string").alias("RELienCreditorKey"),
            target_table.LIE_LENDER.cast("String").alias("RELienCreditorName"),
            target_table.LIE_LENDER_TYPE.cast("String").alias("RELienCreditorType"),
            lit(None).alias("RELienMortgageTypeCode"),
            lit(None).alias("RELienMortgageType"),
            target_table.LIE_VALUE.alias("RELienMortgageAmount"), 
            target_table.LIE_CURRENCY.alias("RELienCurrency"),
            lit(None).alias("RELienComments"),
            target_table.businessdate.alias("BusinessDate"),
        )


        # ================== Joins_fsad ==================
        target_table_fsad = (
            fs_real_estate_investigations.join(
                fsad_real_estate_investigations_properties,
                (
                    fs_real_estate_investigations["REIN_CODE_FS"]
                    == fsad_real_estate_investigations_properties["REIN_CODE"]
                ),
                "inner",
            )
        ).join(
            fsad_real_estate_investigations_properties_liens,
            (
                fsad_real_estate_investigations_properties["REIP_CODE_P"]
                == fsad_real_estate_investigations_properties_liens["REIP_CODE"]
            ),
            "left",
        )

        target_table_fsad_primary_key_fields = [
            target_table_fsad.REIP_CODE,
            target_table_fsad.REPL_UNIQUE_ID,
            lit("qca"),
        ]
        RealEstateInfoKey_fsad_primary_key_fields = [
            target_table_fsad.REIP_CODE,
            target_table_fsad.REPL_UNIQUE_ID,
        ]

        target_table_fsad = target_table_fsad.withColumn(
            "RealEstateInfoId",
            sha2(concat_ws(";", *target_table_fsad_primary_key_fields), 256),
        ).withColumn(
            "RealEstateInfoKey",
            concat_ws(";", *RealEstateInfoKey_fsad_primary_key_fields),
        )

        result_df_fsad = target_table_fsad.select(
            target_table_fsad.RealEstateInfoId,
            lit("qca").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            target_table_fsad.RealEstateInfoKey,
            target_table_fsad.REIP_CODE_P.cast("String").alias("RealEstateKey"),
            target_table_fsad.CUST_CODE.cast("String").alias("OwnerPartyKey"),
            lit(None).alias("OwnershipKey"),
            lit(None).alias("OwnershipReferenceKey"),
            lit(None).alias("OwnershipCode"),
            target_table_fsad.REIP_COOWNERSHIP_TYPE_INTCODE.alias("OwnershipTypeCode"),
            target_table_fsad.REIP_COOWNERSHIP_TYPE_INTCODE.alias("OwnershipType"),
            lit(None).alias("OwnershipPercentage"),
            lit(None).alias("OwnershipStatusCode"),
            lit(None).alias("OwnershipStatus"),
            lit(None).alias("OwnershipStatusDate"),
            target_table_fsad.REPL_DATE.cast("date").alias("NotaryDate"),
            lit(None).alias("NotaryNumber"),
            target_table_fsad.REPL_UNIQUE_ID.cast("int").alias("RELienKey"),
            lit(None).alias("RELienReferenceKey"),
            lit(None).alias("RELienIsActiveFlag"),
            lit(None).alias("RELienStatusCode"),
            lit(None).alias("RELienStatus"),
            target_table_fsad.REPL_ACTIVE.alias("RELienActive"),
            lit(None).alias("RELienTypeCode"),
            lit(None).alias("RELienType"),
            lit(None).alias("RELienReasonCode"),
            lit(None).alias("RELienReason"),
            lit(None).cast("int").alias("RELienInitialOrder"),
            target_table_fsad.REPL_ORDER.cast("int").alias("RELienCurrentOrder"),
            lit(None).alias("RELienCreditorKey"),
            target_table_fsad.REPL_CREDITOR.cast("String").alias("RELienCreditorName"),
            target_table_fsad.REPL_CREDITOR_TYPE.cast("String").alias(
                "RELienCreditorType"
            ),
            lit(None).alias("RELienMortgageTypeCode"),
            lit(None).alias("RELienMortgageType"),
            lit(None).alias("RELienMortgageAmount"),
            lit(None).alias("RELienCurrency"),
            lit(None).alias("RELienComments"),
            target_table_fsad.businessdate.alias("BusinessDate"),
        )

        
        # ================== Append ==================
        final_table = result_df.union(result_df_fsad)
        final_table.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_info_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)